# 실습 10 · RDKit 으로 분자 다루기
### 신약개발 AI 의 필수 기본기 (화학정보학)

**왜 중요한가**
신약개발 AI 는 결국 **분자를 컴퓨터가 이해하는 형태로 다루는** 데서 출발합니다.
RDKit 은 제약사·바이오텍이 표준으로 쓰는 오픈소스 화학정보학 라이브러리입니다.

**이 노트북에서 배우는 것**
- SMILES ↔ 분자 객체 ↔ 구조 그림
- 분자 특성(descriptor) & **Lipinski 5원칙**(약물유사성)
- **분자지문(fingerprint)** 과 **Tanimoto 유사도**
- 부분구조 검색(substructure) — 실제 스크리닝에서 쓰는 기능


In [ ]:
!pip install rdkit -q
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, Crippen, AllChem, DataStructs
from rdkit.Chem.Draw import rdMolDraw2D
import pandas as pd, numpy as np
print("RDKit 준비 완료 ✅")

## 1. SMILES → 분자 → 그림
SMILES 는 분자의 '글자 표현'입니다. 실제 신약들을 그려봅니다.


In [ ]:
drugs = {
    "아스피린"   : "CC(=O)Oc1ccccc1C(=O)O",
    "카페인"     : "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "이부프로펜" : "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "파라세타몰" : "CC(=O)Nc1ccc(O)cc1",
    "페니실린G"  : "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",
    "이마티닙"   : "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1",
}
mols = [Chem.MolFromSmiles(s) for s in drugs.values()]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(230,180),
                     legends=list(drugs.keys()))

## 2. 분자 특성 & Lipinski 5원칙 (약물유사성)
**Lipinski's Rule of Five**: 경구용 약이 되기 좋은 조건 (하나의 경험칙)
- 분자량 ≤ 500
- LogP ≤ 5
- 수소결합 주개 ≤ 5
- 수소결합 받개 ≤ 10

위반이 2개 이상이면 경구흡수가 어려울 수 있습니다. 실제 후보물질 스크리닝의 1차 필터입니다.


In [ ]:
def properties(name, smi):
    m = Chem.MolFromSmiles(smi)
    mw   = Descriptors.MolWt(m)
    logp = Crippen.MolLogP(m)
    hbd  = Descriptors.NumHDonors(m)
    hba  = Descriptors.NumHAcceptors(m)
    viol = sum([mw>500, logp>5, hbd>5, hba>10])   # Lipinski 위반 수
    return {"분자":name,"분자량":round(mw,1),"LogP":round(logp,2),
            "HBD":hbd,"HBA":hba,"Lipinski위반":viol,
            "TPSA":round(Descriptors.TPSA(m),1),
            "회전결합":Descriptors.NumRotatableBonds(m)}

pd.DataFrame([properties(n,s) for n,s in drugs.items()])

## 3. ⭐ 분자지문(Fingerprint)과 유사도
분자의 부분구조 존재 여부를 **0/1 비트열(지문)** 로 표현하면,
두 분자가 얼마나 닮았는지 **Tanimoto 유사도**(0~1)로 계산할 수 있습니다.
→ "비슷한 약은 비슷한 효과" 라는 가정으로 **유사물질 탐색**에 활용 (다음 예제 12의 핵심).


In [ ]:
# Morgan(ECFP) 지문 생성
def fp(smi):
    m = Chem.MolFromSmiles(smi)
    return AllChem.GetMorganFingerprintAsBitVect(m, radius=2, nBits=2048)

fps = {n: fp(s) for n,s in drugs.items()}

# 아스피린과 나머지의 유사도
base = fps["아스피린"]
for n, f in fps.items():
    sim = DataStructs.TanimotoSimilarity(base, f)
    print(f"아스피린 vs {n:10s}  Tanimoto = {sim:.3f}")

In [ ]:
# 전체 쌍 유사도 행렬을 히트맵으로 (구조적으로 닮은 약끼리 묶임)
import matplotlib.pyplot as plt
names = list(fps); M = np.zeros((len(names),len(names)))
for i,a in enumerate(names):
    for j,b in enumerate(names):
        M[i,j] = DataStructs.TanimotoSimilarity(fps[a], fps[b])
plt.figure(figsize=(6,5))
plt.imshow(M, cmap="viridis", vmin=0, vmax=1)
plt.xticks(range(len(names)), names, rotation=45, ha="right")
plt.yticks(range(len(names)), names)
plt.colorbar(label="Tanimoto 유사도")
for i in range(len(names)):
    for j in range(len(names)):
        plt.text(j,i,f"{M[i,j]:.2f}",ha="center",va="center",
                 color="white" if M[i,j]<0.6 else "black", fontsize=8)
plt.title("분자 유사도 행렬"); plt.tight_layout(); plt.show()

## 4. 부분구조 검색 (Substructure Search)
특정 작용기(예: 카복실산 -COOH)를 **포함하는 분자만 골라내기**.
가상 라이브러리에서 원하는 화학적 특징을 가진 분자를 필터링할 때 씁니다.


In [ ]:
# 카복실산 패턴을 SMARTS 로 정의
pattern = Chem.MolFromSmarts("C(=O)[OH]")
print("카복실산(-COOH)을 포함하는 약물:")
for n, s in drugs.items():
    has = Chem.MolFromSmiles(s).HasSubstructMatch(pattern)
    print(f"  {n:10s} : {'있음 ✅' if has else '없음'}")

In [ ]:
# 매칭된 부분구조를 강조해서 그리기 (이부프로펜의 -COOH 강조)
m = Chem.MolFromSmiles(drugs["이부프로펜"])
match = m.GetSubstructMatch(pattern)
d = rdMolDraw2D.MolDraw2DCairo(400, 300)
rdMolDraw2D.PrepareAndDrawMolecule(d, m, highlightAtoms=match)
d.FinishDrawing()
from IPython.display import Image as IPImage
open("hl.png","wb").write(d.GetDrawingText())
IPImage("hl.png")

## 정리 & 현장 응용
- SMILES ↔ 분자 ↔ 구조/특성 자유자재 변환이 **모든 신약 AI 의 출발점**
- **Lipinski** 로 약물유사성 1차 필터, **지문+Tanimoto** 로 유사물질 탐색
- **부분구조 검색**으로 원하는 화학특징 필터링
- 다음 예제: 이 기본기로 **활성 예측(QSAR)** 과 **가상 스크리닝**을 구현
